[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C07_ML_Foundations_Course/06_generalization_metrics/06_generalization_metrics.ipynb)

# 06 · 泛化与指标

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
在真实的 **UCI Adult 收入**（不平衡：只有 24% 的人 >50K）上，从零实现 ROC/PR/AUC、校准/ECE，看指标如何暴露真相。

**你将完成：**
1. 混淆矩阵 → precision/recall/F1，看 accuracy 在不平衡下怎么骗人
2. ROC 曲线 + AUC（从零，含 AUC 的概率解释验证）
3. PR 曲线，理解为什么不平衡该看 PR
4. 校准：ECE + 可靠性图 + 温度缩放

> 数据：UCI Adult（32561 人, 用 6 个数值特征预测收入是否 >50K）。

## 0 · 加载数据 + 训练一个分类器

In [ ]:
import os, urllib.request
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
def trapz(yv, xv):  # np.trapz 在 numpy>=2.0 改名，这里自己实现保证可移植
    xv=np.asarray(xv,float); yv=np.asarray(yv,float)
    return float(np.sum(np.diff(xv)*(yv[:-1]+yv[1:])/2))
CACHE=os.path.expanduser("~/.ml_foundations_data"); os.makedirs(CACHE,exist_ok=True)
def fetch(u,f):
    p=os.path.join(CACHE,f)
    if not os.path.exists(p): urllib.request.urlretrieve(u,p)
    return p

cols=["age","workclass","fnlwgt","education","education_num","marital","occupation",
      "relationship","race","sex","capital_gain","capital_loss","hours_per_week","country","income"]
adf=pd.read_csv(fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data","adult.data"),
                header=None, names=cols, skipinitialspace=True)
num=["age","education_num","capital_gain","capital_loss","hours_per_week","fnlwgt"]
X=adf[num].to_numpy(float); y=(adf["income"]==">50K").astype(float).to_numpy()
print(f"X:{X.shape}  正类(>50K)占比={y.mean():.3f}  -> 不平衡!")

rng=np.random.default_rng(0); idx=rng.permutation(len(y)); cut=int(0.7*len(idx))
tr,te=idx[:cut],idx[cut:]
mu,sd=X[tr].mean(0),X[tr].std(0); Xtr,Xte=(X[tr]-mu)/sd,(X[te]-mu)/sd; ytr,yte=y[tr],y[te]
def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-500,500)))
w=np.zeros(X.shape[1]); b=0.0
for _ in range(800):
    p=sigmoid(Xtr@w+b); g=p-ytr; w-=0.3*Xtr.T@g/len(ytr); b-=0.3*g.mean()
scores=sigmoid(Xte@w+b)
acc=((scores>0.5)==yte).mean(); base=max(yte.mean(),1-yte.mean())
print(f"模型 accuracy={acc:.3f}   majority baseline={base:.3f}")
print("=> accuracy 只比'全猜<=50K'好一点点，但这掩盖了它对正类的能力，下面看真相")

## 1 · 混淆矩阵与 precision/recall

accuracy 高不代表抓得住正类。看混淆矩阵和 precision/recall。

In [ ]:
def confusion(y, pred):
    TP=int(((pred==1)&(y==1)).sum()); FP=int(((pred==1)&(y==0)).sum())
    TN=int(((pred==0)&(y==0)).sum()); FN=int(((pred==0)&(y==1)).sum())
    return TP,FP,TN,FN
pred=(scores>0.5).astype(float)
TP,FP,TN,FN=confusion(yte,pred)
prec=TP/(TP+FP); rec=TP/(TP+FN); f1=2*prec*rec/(prec+rec)
print(f"TP={TP} FP={FP} TN={TN} FN={FN}")
print(f"precision={prec:.3f}  recall={rec:.3f}  F1={f1:.3f}")
print(f"=> 漏掉了 {FN/(TP+FN):.0%} 的高收入者（recall 低），accuracy 完全没暴露这点")

## 2 · ROC 曲线 + AUC（从零）

扫所有阈值得到 (FPR, TPR) 序列，梯形法积分得 AUC。再验证 AUC 的概率解释：
随机正样本分数 > 随机负样本分数的概率。

In [ ]:
def roc_auc(y, scores):
    order=np.argsort(-scores); y=y[order]
    P=y.sum(); N=len(y)-P
    tpr=np.cumsum(y)/P; fpr=np.cumsum(1-y)/N
    tpr=np.concatenate([[0],tpr]); fpr=np.concatenate([[0],fpr])
    auc=trapz(tpr, fpr)
    return fpr, tpr, auc
fpr,tpr,auc=roc_auc(yte, scores)
print(f"ROC-AUC = {auc:.4f}")
# 概率解释验证（抽样）
pos=scores[yte==1]; neg=scores[yte==0]
s=rng.choice(pos,5000); t=rng.choice(neg,5000)
prob=(s>t).mean()+0.5*(s==t).mean()
print(f"P(正样本分>负样本分) ≈ {prob:.4f}  ≈ AUC ✓ (AUC 就是排序正确的概率)")

## 3 · PR 曲线：不平衡数据的真相

正类只占 24%。ROC 看着不错，但 PR 曲线揭示在高 recall 时 precision 会崩。

In [ ]:
def pr_curve(y, scores):
    order=np.argsort(-scores); y=y[order]
    tp=np.cumsum(y); fp=np.cumsum(1-y)
    prec=tp/(tp+fp); rec=tp/y.sum()
    return rec, prec
rec_c, prec_c = pr_curve(yte, scores)
# PR-AUC（average precision 近似）
ap=trapz(prec_c, rec_c)
print(f"PR-AUC ≈ {ap:.4f}  （随机基线 = 正类比例 = {yte.mean():.3f}）")
print(f"在 recall=0.8 处 precision ≈ {prec_c[np.argmin(np.abs(rec_c-0.8))]:.3f}")
print("=> ROC-AUC 0.8+ 看着很好，但要抓住 80% 高收入者，precision 掉得明显")

## 4 · 校准：ECE + 温度缩放

模型概率可信吗？分桶比较"平均预测概率"vs"实际正类比例"。再用温度缩放改善。

In [ ]:
def ece(y, probs, n_bins=10):
    bins=np.linspace(0,1,n_bins+1); e=0.0
    rows=[]
    for i in range(n_bins):
        m=(probs>bins[i])&(probs<=bins[i+1])
        if m.sum()==0: continue
        conf=probs[m].mean(); acc=y[m].mean()
        e+=m.sum()/len(y)*abs(acc-conf)
        rows.append((round(conf,2),round(acc,2),int(m.sum())))
    return e, rows
e0,rows=ece(yte, scores)
print(f"ECE = {e0:.4f}")
print("桶(平均置信, 实际正类率, 数量) 前几:", rows[:4])

# 温度缩放：在 logit 上除以 T（这里直接搜一个 T 最小化 ECE 演示）
logit=np.log(scores/(1-scores)+1e-12)
best=min(np.arange(0.5,3.01,0.1), key=lambda T: ece(yte, sigmoid(logit/T))[0])
e1,_=ece(yte, sigmoid(logit/best))
print(f"最优温度 T={best:.1f}  缩放后 ECE={e1:.4f}  (改善={e0-e1:+.4f})")

## 5 · 阈值不是 0.5，校准不改排序

两个评测里反复要解释的点，在真实 Adult 模型上看清楚。**(a) 0.5 不是金科玉律**：决策阈值是一个独立于模型的旋钮，应按指标或业务代价来选。下面扫一遍阈值，会发现 F1 最优阈值并不在 0.5；如果漏掉一个高收入者（FN）比误判（FP）贵 5 倍，代价最优阈值还会进一步往低调（换取更高 recall）。**(b) 温度缩放只动校准、不动排序**：给 logit 除以一个温度 $T$ 是单调变换，不改变样本分数的相对顺序，所以 AUC 一字不变，但概率尺度变了、ECE 随之改变——这正是「排序好（高 AUC）≠ 校准好（低 ECE）」的直接证据。

In [ ]:
# (a) 0.5 不是金科玉律：扫阈值，找 F1 最优阈值，并按业务代价（漏诊更贵）选阈值
def metrics_at(thr):
    pred = (scores > thr).astype(float)
    TP = ((pred==1)&(yte==1)).sum(); FP = ((pred==1)&(yte==0)).sum()
    FN = ((pred==0)&(yte==1)).sum()
    p = TP/(TP+FP) if TP+FP>0 else 0.0
    r = TP/(TP+FN) if TP+FN>0 else 0.0
    f1 = 2*p*r/(p+r) if p+r>0 else 0.0
    return p, r, f1, int(FP), int(FN)

ths = np.linspace(0.05, 0.95, 91)
f1s = [metrics_at(t)[2] for t in ths]
t_best = ths[int(np.argmax(f1s))]
p05, r05, f05, *_ = metrics_at(0.5)
pb, rb, fb, *_ = metrics_at(t_best)
print(f"阈值 0.50      : precision={p05:.3f} recall={r05:.3f} F1={f05:.3f}")
print(f"F1 最优阈值 {t_best:.2f} : precision={pb:.3f} recall={rb:.3f} F1={fb:.3f}")
print(f"=> F1 最优阈值是 {t_best:.2f} 而非 0.5（不平衡数据上默认 0.5 通常次优）")

# 按代价选阈值：假设漏掉一个高收入者(FN)的代价是误判一个(FP)的 5 倍
cost = [(t, 5*metrics_at(t)[4] + 1*metrics_at(t)[3]) for t in ths]
t_cost = min(cost, key=lambda x: x[1])[0]
print(f"代价最优阈值（FN:FP=5:1）= {t_cost:.2f}  -> 更怕漏诊就把阈值调低（多召回）")
assert abs(t_best - 0.5) > 1e-6, "F1 最优阈值不应恰好是 0.5"
assert t_cost <= t_best + 1e-9, "漏诊更贵时，代价最优阈值应不高于 F1 最优阈值（倾向更高 recall）"

# (b) 温度缩放只改概率尺度、不改排序：AUC 不变，ECE 改变
fpr0, tpr0, auc0 = roc_auc(yte, scores)
logit = np.log(scores/(1-scores)+1e-12)
scaled = sigmoid(logit/2.0)                       # 任取一个温度 T=2
fpr1, tpr1, auc1 = roc_auc(yte, scaled)
e_before, _ = ece(yte, scores); e_after, _ = ece(yte, scaled)
print(f"\n温度缩放 T=2:  AUC {auc0:.4f} -> {auc1:.4f}（几乎不变，排序不动）   ECE {e_before:.4f} -> {e_after:.4f}（改变了）")
assert abs(auc0 - auc1) < 1e-6, "单调的温度缩放不改变排序，故 AUC 不变"
print("=> 验证『排序好(高 AUC) ≠ 校准好(低 ECE)』：温度缩放是只动校准、不动排序的事后手术")

---
## ✏️ 练习区

### ✏️ 练习 1：从混淆矩阵到 precision/recall/F1

实现 `prf1(y, pred)` 返回 `(precision, recall, f1)`。处理分母为 0（返回 0）。

In [ ]:
def prf1(y, pred):
    # TODO: 算 TP/FP/FN -> precision, recall, f1（分母0时该项为0）
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
yt=np.array([1,1,0,0,1]); pr=np.array([1,0,0,1,1])
p,r,f=prf1(yt,pr)
assert abs(p-2/3)<1e-9 and abs(r-2/3)<1e-9 and abs(f-2/3)<1e-9
p,r,f=prf1(yt, np.zeros(5))   # 全预测0
assert p==0 and r==0 and f==0
print("练习 1 通过 ✓")


### ✏️ 练习 2：ROC-AUC 从零

实现 `compute_auc(y, scores)` 只返回 AUC 值。要求与"概率解释"一致。

In [ ]:
def compute_auc(y, scores):
    # TODO: 按分数降序累计 TPR/FPR，梯形积分；或用 rank 公式
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
auc_mine=compute_auc(yte, scores)
assert abs(auc_mine - auc) < 1e-3
# 完美分类器 AUC=1；随机分数 AUC≈0.5
assert abs(compute_auc(yte, yte.astype(float)) - 1.0) < 1e-9
rand_auc=compute_auc(yte, rng.random(len(yte)))
assert abs(rand_auc-0.5) < 0.05
print(f"练习 2 通过 ✓  AUC={auc_mine:.4f}")


### ✏️ 练习 3：ECE（期望校准误差）

实现 `compute_ece(y, probs, n_bins)`：等宽分桶，返回 ECE 标量。

In [ ]:
def compute_ece(y, probs, n_bins=10):
    # TODO: 分桶，每桶 |实际正类率 - 平均置信| 按样本数加权求和
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
assert abs(compute_ece(yte, scores) - e0) < 1e-9
# 完美校准的概率（用真实频率）ECE 应很小
perfect=np.where(yte==1, 0.9, 0.1)  # 故意制造可校准信号
assert compute_ece(yte, perfect) >= 0
# 全 0.5 的概率，ECE = |正类率 - 0.5|
assert abs(compute_ece(yte, np.full(len(yte),0.5)) - abs(yte.mean()-0.5)) < 1e-9
print(f"练习 3 通过 ✓  模型 ECE={compute_ece(yte,scores):.4f}")


### ✏️ 练习 4：bias-variance 实测（bootstrap）

在真实数据上：对同一测试点，训练 `n_models` 个 bootstrap 逻辑回归，
实现 `bias_variance(Xtr,ytr,Xte,n_models)` 返回测试集上预测概率的**平均 variance**。
验证：增大 L2 正则会降低 variance。

In [ ]:
def fit_lr(X,y,l2=0.0,steps=300,lr=0.3):
    w=np.zeros(X.shape[1]); b=0.0
    for _ in range(steps):
        p=sigmoid(X@w+b); g=p-y; w-=lr*(X.T@g/len(y)+l2*w); b-=lr*g.mean()
    return w,b

def bias_variance(Xtr, ytr, Xte, n_models=20, l2=0.0, seed=0):
    # TODO: bootstrap 训练 n_models 个模型，收集每个对 Xte 的预测概率，
    #       返回这些预测在每个测试点上的 variance 的平均值
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
v_low = bias_variance(Xtr, ytr, Xte, n_models=15, l2=0.0,  seed=1)
v_high= bias_variance(Xtr, ytr, Xte, n_models=15, l2=2.0,  seed=1)
assert v_high < v_low, "更强 L2 应降低预测方差"
print(f"练习 4 通过 ✓  variance: 无正则={v_low:.2e}  强正则={v_high:.2e}（正则降方差）")


---
## 📖 参考答案

In [ ]:
# 练习 1
def prf1(y, pred):
    TP=((pred==1)&(y==1)).sum(); FP=((pred==1)&(y==0)).sum(); FN=((pred==0)&(y==1)).sum()
    p=TP/(TP+FP) if TP+FP>0 else 0.0
    r=TP/(TP+FN) if TP+FN>0 else 0.0
    f=2*p*r/(p+r) if p+r>0 else 0.0
    return p,r,f
print("练习 1 ✓")

In [ ]:
# 练习 2
def compute_auc(y, scores):
    order=np.argsort(-scores); y=y[order]; P=y.sum(); N=len(y)-P
    tpr=np.concatenate([[0],np.cumsum(y)/P]); fpr=np.concatenate([[0],np.cumsum(1-y)/N])
    return trapz(tpr,fpr)
print("练习 2 ✓")

In [ ]:
# 练习 3
def compute_ece(y, probs, n_bins=10):
    bins=np.linspace(0,1,n_bins+1); e=0.0
    for i in range(n_bins):
        m=(probs>bins[i])&(probs<=bins[i+1]) if i>0 else (probs>=bins[0])&(probs<=bins[1])
        if m.sum()==0: continue
        e+=m.sum()/len(y)*abs(y[m].mean()-probs[m].mean())
    return float(e)
print("练习 3 ✓")

In [ ]:
# 练习 4
def bias_variance(Xtr, ytr, Xte, n_models=20, l2=0.0, seed=0):
    rng=np.random.default_rng(seed); preds=[]
    for _ in range(n_models):
        idx=rng.integers(0,len(ytr),len(ytr))
        w,b=fit_lr(Xtr[idx],ytr[idx],l2=l2)
        preds.append(sigmoid(Xte@w+b))
    return float(np.var(np.array(preds),axis=0).mean())
print("练习 4 ✓ —— 你刚在真实数据上实测了 bias-variance 权衡")